# Export Network as Touchstone File

Frequently a `Network` is required to be saved as a [Touchstone file](https://en.wikipedia.org/wiki/Touchstone_file) so that it can be imported into another program such as [Keysight ADS](https://www.keysight.com/zz/en/products/software/pathwave-design-software/pathwave-advanced-design-system.html) or [Microwave Office](https://www.cadence.com/en_US/home/tools/system-analysis/rf-microwave-design/awr-design-environment-platform.html). The `Network.write_touchstone()` method allows `Network` objects to be exported in the appropriate format.

In [1]:
from pathlib import Path

import numpy as np

import skrf as rf

In this example, we are going to first generate a random 2-port `Network` and export the data as a Touchstone file using all the default arguments

In [2]:
# Handle for random generator
rand_gen = np.random.default_rng()

# Set the number of points
ports = 2

# Generate the frequency object
points = 11
freq = rf.Frequency(1, 50, points, "GHz")

# Create random data to fill the Network
s_random = rand_gen.uniform(-1, 1, (freq.npoints, ports, ports)) + 1j * rand_gen.uniform(
    -1, 1, (freq.npoints, ports, ports)
)

# Generate the Network object
ntwk = rf.Network(s=s_random, frequency=freq, name="test_ntwk")
ntwk.comments = "Default output"

# Define the save location
filepath = Path(f"test_network.s{ports}p")

# Save the Network as a touchstone file using the default arguments
print(ntwk.write_touchstone(filepath, return_string=True))

!Default output
! Created with skrf 2.2.0 (http://scikit-rf.org).
# GHz S RI R 50.0 
!freq ReS11 ImS11 ReS21 ImS21 ReS12 ImS12 ReS22 ImS22
1.0 0.9510827982807804 0.433461930535483 0.24705208741926588 -0.5831644267975584 0.022599338777870237 -0.3203291548009106 0.4675417257721579 -0.48657171271747823
5.9 -0.013176338702732826 -0.44055930615392325 0.06011608746909358 0.1609462829737327 -0.7174863390244903 -0.3716487212094355 0.509433924957297 -0.6095676868639011
10.8 -0.48819309311899306 -0.7989027599986307 0.9225912312417057 0.41672014919694744 -0.1292840064435674 -0.4337659853351916 0.2988287028207288 0.9719666103737614
15.7 0.2605048914613852 0.5575447656283614 -0.5264228330934944 0.43453022677923014 -0.2729693612855535 0.5009839770546003 0.7336965812440237 0.8802532068342483
20.6 0.1324406675996792 -0.3976347902325408 0.5105002216525507 0.39558947876426975 -0.21982672757912636 -0.9089674402116656 -0.2811468035230944 0.8624590660331488
25.5 0.5113278050625578 -0.8893617490257819 0.509

In this example, we are going to export the same 2-port `Network` as a Touchstone file, but use the formatting strings to format the columns of the resulting file.

In [3]:
# Get a copy of the network
ntwk_formatted = ntwk.copy()
ntwk_formatted.comments = "Formatted output"

# Define the save location
formatted_filepath = Path(f"test_network_formatted.s{ports}p")

# Save the Network as a touchstone file using the string formatting options
freq_fmt = "{:<8.3f}"
data_fmt = "\t{:>8.3f}"
print(
    ntwk_formatted.write_touchstone(
        formatted_filepath,
        form="db",
        return_string=True,
        format_spec_freq=freq_fmt,
        format_spec_A=data_fmt,
        format_spec_B=data_fmt,
    )
)

!Formatted output
! Created with skrf 2.2.0 (http://scikit-rf.org).
# GHz S DB R 50.0 
!freq dBS11 angS11 dBS21 angS21 dBS12 angS12 dBS22 angS22
1.000    	   0.384 	  24.501 	  -3.967 	 -67.041 	  -9.867 	 -85.964 	  -3.417 	 -46.143
5.900    	  -7.116 	 -91.713 	 -15.299 	  69.519 	  -1.851 	-152.616 	  -1.999 	 -50.113
10.800   	  -0.572 	-121.428 	   0.107 	  24.308 	  -6.885 	-106.597 	   0.145 	  72.910
15.700   	  -4.217 	  64.956 	  -3.317 	 140.462 	  -4.875 	 118.584 	   1.183 	  50.189
20.600   	  -7.553 	 -71.579 	  -3.798 	  37.772 	  -0.582 	-103.595 	  -0.847 	 108.055
25.500   	   0.222 	 -60.104 	  -4.910 	  26.172 	  -2.103 	-141.243 	  -2.455 	-106.549
30.400   	   0.021 	 107.268 	  -1.931 	 171.543 	  -5.038 	  97.080 	  -0.258 	 -45.185
35.300   	  -5.456 	  71.435 	  -3.597 	 -93.130 	  -3.224 	 -67.025 	  -1.917 	  17.147
40.200   	   0.135 	 -66.517 	  -5.127 	  19.279 	  -0.617 	 -11.491 	   0.402 	 -54.128
45.100   	  -7.735 	-176.755 	  -1.473 	 -37.545 	   0

In this example, we are going to export a new 2-port `Network` as a Touchstone file, but we're going to include random noise data and format those output columns as well.

In [4]:
# Create random data to fill the Network
s_random = rand_gen.uniform(-1, 1, (freq.npoints, ports, ports)) + 1j * rand_gen.uniform(
    -1, 1, (freq.npoints, ports, ports)
)

# Create random noise data to fill the Network
nfmin_db_random = rand_gen.uniform(0, 10, freq.npoints)
g_opt_random = rand_gen.uniform(-1, 1, freq.npoints) + 1j * rand_gen.uniform(-1, 1, freq.npoints)
rn_random = rand_gen.uniform(size=freq.npoints)

# Generate the Network object
noisy_ntwk = rf.Network(s=s_random, frequency=freq, name="test_noisy_ntwk")
noisy_ntwk.comments = "Formatted output with noise data"

# Add the noise data
noisy_ntwk.set_noise_a(noise_freq=freq, nfmin_db=nfmin_db_random, gamma_opt=g_opt_random, rn=rn_random)

# Define the save location
formatted_filepath = Path(f"test_noisy_network_formatted.s{ports}p")

# Save the Network as a touchstone file using the string formatting options
print(
    noisy_ntwk.write_touchstone(
        formatted_filepath,
        form="db",
        return_string=True,
        format_spec_freq=freq_fmt,
        format_spec_A=data_fmt,
        format_spec_B=data_fmt,
        format_spec_nf_freq=freq_fmt,
        format_spec_nf_min=data_fmt,
        format_spec_g_opt_mag=data_fmt,
        format_spec_g_opt_phase=data_fmt,
        format_spec_rn=data_fmt,
    )
)

!Formatted output with noise data
! Created with skrf 2.2.0 (http://scikit-rf.org).
# GHz S DB R 50.0 
!freq dBS11 angS11 dBS21 angS21 dBS12 angS12 dBS22 angS22
1.000    	   2.035 	  51.789 	  -5.307 	   8.962 	  -5.629 	  27.610 	 -19.936 	-147.544
5.900    	   1.228 	  47.023 	  -2.982 	 -78.642 	  -2.133 	-117.513 	  -4.080 	 -58.168
10.800   	   0.160 	  14.731 	  -8.083 	  84.873 	   1.142 	-130.239 	  -3.981 	-149.235
15.700   	 -15.424 	-120.929 	  -6.275 	  13.193 	  -4.546 	 -97.900 	  -0.971 	   2.360
20.600   	  -2.083 	 -90.624 	  -1.277 	 156.764 	  -8.146 	  33.535 	 -11.334 	  10.688
25.500   	   2.084 	-131.128 	  -4.475 	  65.777 	  -1.064 	 -68.091 	  -0.031 	-135.515
30.400   	  -1.750 	 167.073 	 -14.768 	-161.968 	  -3.220 	 108.053 	   1.931 	 -39.895
35.300   	  -2.101 	  33.419 	  -9.001 	-169.559 	  -6.698 	-161.061 	  -1.965 	 -21.958
40.200   	  -3.140 	  78.247 	  -0.167 	  16.007 	   1.569 	 -53.463 	  -1.248 	-136.729
45.100   	 -15.006 	  98.344 	 -15.354